# IncentiveScope: PyTorch R30 retention research

## tl;dr

Can pre-cutoff trading behaviour rank later R30 participation? This companion reads the saved, frozen-data study. The summary below is computed from its results rather than prewritten to favour the neural network.

Default execution reads local results and figures. Set `INCENTIVESCOPE_RERUN_ML=1` explicitly to rebuild features and train in the separate ML environment; rerun outputs go to ignored `reports/live/ml-rerun`.

In [ ]:
import json
import os
import subprocess
import sys
from pathlib import Path
from IPython.display import Markdown, SVG, display

ROOT = Path.cwd().resolve()
if not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
assert (ROOT / "pyproject.toml").exists(), "Run from the repository or notebooks directory."
sys.path.insert(0, str(ROOT / "src"))
REPORT = ROOT / "reports/gmx-stip/ml"

if os.environ.get("INCENTIVESCOPE_RERUN_ML") == "1":
    ml_python = ROOT / ".venv-ml" / ("Scripts/python.exe" if os.name == "nt" else "bin/python")
    REPORT = ROOT / "reports/live/ml-rerun"
    subprocess.run([
        str(ml_python if ml_python.exists() else Path(sys.executable)),
        "-m", "incentivescope", "ml", "--config", str(ROOT / "configs/gmx-stip-ml.json"),
        "--input", str(ROOT / "data/raw/gmx-stip/trades.csv"),
        "--manifest", str(ROOT / "data/raw/gmx-stip/manifest.json"),
        "--output", str(REPORT), "--device", "cpu",
    ], cwd=ROOT, check=True)

with (REPORT / "results.json").open(encoding="utf-8") as stream:
    results = json.load(stream)
from incentivescope.ml_report import findings
primary = next(row for row in results["models"] if row["name"] == "pytorch_mlp")
display(Markdown("\n\n".join("- " + message for message in findings(results))))

## Context & Methods

The observation is **one address × one prediction cutoff T**. Eligible accounts have a successful positive-size opening/increase during `[earning start, T)`. Features use only `timestamp < T`; the label is an opening/increase during `[T+23 days, T+30 days)` in UTC.

Training labels finish before validation begins; validation labels finish before the final test cutoff. Validation selects model settings and early-stopping epochs and is never merged back into parameter training. Seed 42 is the prespecified primary MLP run; seeds 17 and 73 remain visible.

### Key Assumptions

- This is one campaign, with training under rebates and final labels after rebates end. Predictive associations do not identify incentive causality.
- Repeated addresses at different training cutoffs are allowed; test addresses are also split by participation in parameter training.
- First observed history is bounded by the September 2023 extract. Addresses are not people.
- A later frozen indexer snapshot supports an event-time backtest; historical point-in-time indexing delays and corrections were not audited.
- No additional probability calibration is fitted. Dune agreement remains unestablished.

The MLP is `17 → 32 → 16 → 1` with ReLU, dropout 0.1, AdamW and unweighted BCEWithLogitsLoss. Compare it with training prevalence, recency ranking, Logistic Regression and HistGradientBoosting. Read [the implementation](../src/incentivescope/ml_models.py) and [the configuration](../configs/gmx-stip-ml.json) for exact recorded settings.

## Data

### Frozen inputs and temporal cutoffs

Raw input and caches are intentionally excluded from Git. To train again, restore the frozen release snapshot using the repository README, with its manifest and matching SHA-256. The default notebook requires only committed result JSON and figures.

In [ ]:
from incentivescope.ml_report import label

def markdown_table(headers, rows):
    escape = lambda value: str(value).replace("|", "&#124;").replace("\n", " ")
    lines = ["| " + " | ".join(headers) + " |", "| " + " | ".join("---" for _ in headers) + " |"]
    lines += ["| " + " | ".join(escape(value) for value in row) + " |" for row in rows]
    display(Markdown("\n".join(lines)))

print("Input SHA-256:", results["dataset"]["sha256"])
print("Coverage:", results["dataset"]["coverage_start"], "to", results["dataset"]["coverage_end"], "(exclusive)")
print("Source:", results["dataset"]["source_url"])
markdown_table(["Use", "T (UTC)", "Label start", "Label end (exclusive)", "Samples", "Returns"], [
    [row["split"], row["as_of"], row["label_start"], row["label_end"], row["n"], row["positive"]]
    for row in results["split_summary"]
])
assert primary["test"]["n"] == 22145
assert primary["test"]["positives"] == 954
assert len(results["feature_names"]) == 17
print("Reference cohort reproduced: 954 / 22,145; 17 predictor columns.")

### Feature dictionary and reproducibility record

Counts, money and durations use `log1p`; standardisation uses training examples only. Zero-activity windows remain zero. Positive-size opening fees must be available. Account addresses, event hashes, cutoffs and labels are tracking fields, not features.

In [ ]:
from incentivescope.ml_report import feature_description
markdown_table(["Feature", "Definition", "log1p"], [
    [name, feature_description(name), name in results["log_features"]]
    for name in results["feature_names"]
])
print("Training unique addresses:", results["training_unique_accounts"])
print(json.dumps(results.get("provenance", {}), indent=2))

## Results

### Model comparison

AP is computed from all predictions according to `average_precision_score`; it is not trapezoidal PR-AUC. Model selection uses validation AP. A better final-test result is descriptive evidence, without a claimed significance test.

In [ ]:
markdown_table(["Model", "Validation AP", "Test AP", "ROC-AUC", "Top 10% precision", "Top 10% recall", "Lift", "Brier", "Log loss"], [
    [label(row["name"]), *[round(value, 4) if value is not None else "N/A" for value in [
        row["validation"]["average_precision"], row["test"]["average_precision"], row["test"]["roc_auc"],
        row["test"]["top10_precision"], row["test"]["top10_recall"], row["test"]["top10_lift"],
        row["test"]["brier"], row["test"]["log_loss"]]]]
    for row in results["models"]
])
display(SVG(filename=str(REPORT / "figures/model-comparison.svg")))

### Precision–recall and probability reliability

Curve display points may be bounded; AP still uses the full prediction set. Reliability plots compare the primary MLP with a validation-selected probability baseline. Occupied bins display their sample counts; sparse bins should be interpreted cautiously. The recency method has ranking scores rather than probabilities.

In [ ]:
display(SVG(filename=str(REPORT / "figures/precision-recall.svg")))

In [ ]:
display(SVG(filename=str(REPORT / "figures/reliability.svg")))
markdown_table(["Primary MLP mean probability", "Observed return share", "Bin N"], [
    [round(row["mean_probability"], 4), round(row["observed_rate"], 4), row["count"]]
    for row in primary["test"]["reliability"]
])

### Optimisation and fixed-seed variation

Epochs are selected on validation AP, not final-test results. The three seed results describe optimisation variability; they are not a statistical confidence interval.

In [ ]:
display(SVG(filename=str(REPORT / "figures/training.svg")))
markdown_table(["Seed", "Best epoch", "Validation AP", "Test AP", "Test Brier"], [
    [row["seed"], row["best_epoch"], round(row["validation"]["average_precision"], 4),
     round(row["test"]["average_precision"], 4), round(row["test"]["brier"], 4)]
    for row in results["mlp_seed_results"]
])

### Interpreting feature dependence

Permutation importance shuffles each feature on the validation set for the primary MLP seed 42. Error bars show the standard deviation across ten shuffles, not confidence intervals. Correlated features can share importance; the values do not establish causal effects.

In [ ]:
display(SVG(filename=str(REPORT / "figures/importance.svg")))

### Seen and unseen training addresses

“New to training” means absent from parameter-training cutoffs, and may still include addresses present in validation. This is not a claim of newly acquired traders or unique people.

In [ ]:
markdown_table(["Model", "Training membership", "Returns / N", "AP", "Top 10% lift"], [
    [label(model["name"]), group, str(value["positives"]) + "/" + str(value["n"]),
     round(value["average_precision"], 4), round(value["top10_lift"], 4)]
    for model in results["models"] for group, value in model["subgroups"].items()
])

### Editable plotting code (opt-in)

All five research figures are generated by `figures_ml` in `src/incentivescope/ml_report.py`, using standard Matplotlib from the saved result rows. Edit that function to change labels or styling, then explicitly set `INCENTIVESCOPE_RERENDER_ML=1` to render fresh SVG/PNG files into ignored `reports/live/ml-figures`. Default notebook execution only displays the reference files.

In [ ]:
if os.environ.get("INCENTIVESCOPE_RERENDER_ML") == "1":
    from incentivescope.ml_report import figures_ml
    edited_assets = figures_ml(results, ROOT / "reports/live/ml-figures")
    display(SVG(filename=str(edited_assets["model-comparison"])))
else:
    print("Plot renderer is editable in src/incentivescope/ml_report.py; reference figures were read without rewriting them.")

## Takeaways

The comparison must be interpreted with the shift from training under rebates to testing after rebates. A simple model can be preferable if its ranking or probability quality matches or exceeds the MLP. The project reports all results without requiring a neural-network victory. Full model-specific bins, subgroup metrics and settings remain inspectable in the JSON and portable HTML.

In [ ]:
from incentivescope.ml_report import LIMITATIONS
display(Markdown("\n".join("- " + text for text in LIMITATIONS)))
print("Saved study:", REPORT / "index.html")